# DENTEX Dental AI Pipeline (Kaggle Notebook)

This notebook implements a Kaggle-ready, two-phase pipeline for DENTEX:
1. SimMIM pretraining on unlabeled panoramic X-rays.
2. Hierarchical supervised training with shared Swin+FPN, quadrant-first processing, parallel U-Net and DINO branches, and WBF+Hungarian postprocessing.

The notebook uses a `quick` runtime profile for smoke testing and a `full` profile for benchmark-length runs.

> NOTE: Pathology detection is intentionally out of scope for this version and is included only as a placeholder class.


In [ ]:
# Section 1: Runtime Setup and Profile Switch
# Install packages expected in Kaggle runtimes.
%pip install timm==0.9.16 einops ensemble-boxes scipy scikit-image albumentations pycocotools wandb --quiet


In [ ]:
# Section 1: Runtime Setup and Profile Switch

from __future__ import annotations

import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import wandb
from albumentations.pytorch import ToTensorV2
from ensemble_boxes import weighted_boxes_fusion
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from scipy.optimize import linear_sum_assignment
from skimage.measure import label, regionprops
from torch.amp import GradScaler, autocast
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Sampler
from torchvision.ops import generalized_box_iou, roi_align
from tqdm.auto import tqdm


def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


RUNTIME_PROFILE: str = os.environ.get("DENTEX_PROFILE", "quick").strip().lower()
assert RUNTIME_PROFILE in {"quick", "full"}, "RUNTIME_PROFILE must be 'quick' or 'full'."

print(f"Using runtime profile: {RUNTIME_PROFILE}")


def ensure_dir(path: str | Path) -> Path:
    """Create and return a directory path."""
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


# Kaggle-specific output root.
OUTPUT_ROOT = ensure_dir("/kaggle/working/outputs")


# Weights and Biases login using Kaggle Secrets when available.
def login_wandb_from_kaggle_secret(secret_name: str = "wandb_api") -> bool:
    """Attempt W&B login from Kaggle Secrets. Returns True if login succeeds."""
    try:
        from kaggle_secrets import UserSecretsClient

        user_secrets = UserSecretsClient()
        wandb_api_key = user_secrets.get_secret(secret_name)
        wandb.login(key=wandb_api_key)
        print("W&B login succeeded via Kaggle Secrets.")
        return True
    except Exception as exc:
        print(f"W&B login skipped: {exc}")
        return False


WANDB_READY = login_wandb_from_kaggle_secret()


## 2. Unified Configuration and Validators

This section defines a single typed configuration object. All constants used later are sourced from this object.


In [ ]:
@dataclass
class DentexConfig:
    """Typed configuration for all notebook constants and training knobs."""

    # Paths
    data_root: str = "/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023"
    output_dir: str = "/kaggle/working/outputs"

    # W&B
    wandb_project: str = "dentex-pipeline"
    wandb_run_name: str = "swin-unet-dino-run1"

    # Image
    img_w: int = 1280
    img_h: int = 640
    image_channels: int = 3

    # Backbone
    swin_model: str = "swin_base_patch4_window7_224"
    swin_pretrained: bool = True
    fpn_out_channels: int = 256
    fpn_levels: Tuple[str, str, str, str] = ("P2", "P3", "P4", "P5")
    fpn_out_indices: Tuple[int, int, int, int] = (0, 1, 2, 3)

    # SimMIM
    simmim_mask_ratio: float = 0.60
    simmim_patch_size: int = 32
    simmim_epochs: int = 50
    simmim_lr: float = 1e-4
    simmim_weight_decay: float = 0.05
    simmim_batch_size: int = 8

    # Main training
    train_iterations: int = 40000
    train_batch_size: int = 4
    optimizer: str = "adamw"
    lr: float = 2.5e-5
    weight_decay: float = 1e-4
    lr_scheduler: str = "cosine"
    warmup_steps: int = 1000
    grad_clip_max_norm: float = 1.0
    amp: bool = True

    # Sampling ratios
    ratio_diagnosis: float = 0.40
    ratio_quad_enum: float = 0.30
    ratio_quad_only: float = 0.30

    # Quadrant head
    num_quadrants: int = 4
    quad_conf_thresh: float = 0.5
    quad_loss_weight_box: float = 2.0
    quad_loss_weight_cls: float = 1.0

    # Segmentation
    seg_num_classes: int = 1
    seg_unet_channels: Tuple[int, int, int, int] = (256, 128, 64, 32)
    seg_bce_weight: float = 0.5
    seg_dice_weight: float = 0.5
    min_component_area: int = 50

    # DINO
    dino_num_queries: int = 32
    dino_num_decoder_layers: int = 6
    dino_hidden_dim: int = 256
    dino_num_heads: int = 8
    dino_ffn_dim: int = 1024
    dino_dropout: float = 0.1
    num_fdi_classes: int = 32
    num_positions_per_quadrant: int = 8
    dino_num_output_classes: int = 9  # 8 positions + background
    dino_lambda_box: float = 5.0
    dino_lambda_giou: float = 2.0
    dino_lambda_cls: float = 1.0

    # CoordConv
    use_coordconv: bool = True

    # Post-processing
    wbf_iou_thresh: float = 0.55
    wbf_score_thresh: float = 0.01
    hungarian_iou_thresh: float = 0.50

    # Evaluation
    iou_thresholds: Tuple[float, ...] = (0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95)

    # Runtime
    seed: int = 42
    num_workers: int = 2
    pin_memory: bool = True
    persistent_workers: bool = False

    # Checkpointing and logging
    val_every_iters: int = 500
    ckpt_every_iters: int = 2000

    # Numerical helpers
    eps: float = 1e-6

    # Quick profile overrides (iteration-heavy only)
    quick_overrides: Dict[str, Any] = field(default_factory=lambda: {
        "simmim_batch_size": 1,
        "train_batch_size": 1,
        "num_workers": 0,
        "simmim_epochs": 1,
        "train_iterations": 100,
        "val_every_iters": 100,
        "ckpt_every_iters": 250,
    })

    def validate(self) -> None:
        """Validate config values and structural assumptions."""
        assert self.img_w > 0 and self.img_h > 0, "Image size must be positive."
        assert self.num_quadrants == 4, "DENTEX quadrant stage expects exactly 4 quadrants."
        assert abs(self.ratio_diagnosis + self.ratio_quad_enum + self.ratio_quad_only - 1.0) < self.eps
        assert self.optimizer.lower() == "adamw", "Notebook currently supports AdamW as specified."
        assert self.lr_scheduler.lower() == "cosine", "Notebook currently supports cosine scheduler."
        assert self.dino_num_output_classes == self.num_positions_per_quadrant + 1
        assert Path(self.output_dir).as_posix().startswith("/kaggle/working"), "Output dir must be under /kaggle/working for Kaggle."


def apply_runtime_profile(cfg: DentexConfig, profile: str) -> DentexConfig:
    """Apply runtime profile overrides without changing architecture constants."""
    profile = profile.lower().strip()
    assert profile in {"quick", "full"}
    if profile == "quick":
        for key, value in cfg.quick_overrides.items():
            assert hasattr(cfg, key), f"Unknown quick override key: {key}"
            setattr(cfg, key, value)
    cfg.validate()
    return cfg


CFG = apply_runtime_profile(DentexConfig(), RUNTIME_PROFILE)
set_seed(CFG.seed)
ensure_dir(CFG.output_dir)

print("Configuration loaded.")
print(json.dumps(asdict(CFG), indent=2, default=str)[:2000])


## 3. Dataset Discovery and Kaggle Path Guards

This section validates Kaggle directory variants and resolves JSON/image roots for all three labeled subsets.


In [ ]:
import re


def find_existing_path(candidates: Sequence[Path]) -> Path:
    """Return the first existing path from candidates, else raise with details."""
    for path in candidates:
        if path.exists():
            return path
    candidate_text = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(f"None of the expected paths exist:\n{candidate_text}")


def resolve_dentex_paths(cfg: DentexConfig) -> Dict[str, Dict[str, Any]]:
    """Resolve dataset subset paths under Kaggle input with defensive path variants."""
    root = Path(cfg.data_root)
    train_root = find_existing_path([
        root / "training_data" / "training_data",
        root / "training_data",
    ])

    specs = {
        "quadrant": {
            "dir": "quadrant",
            "json": "train_quadrant.json",
            "label_level": "quadrant",
        },
        "quadrant_enumeration": {
            "dir": "quadrant_enumeration",
            "json": "train_quadrant_enumeration.json",
            "label_level": "quadrant_enumeration",
        },
        "quadrant_enumeration_disease": {
            "dir": "quadrant-enumeration-disease",
            "json": "train_quadrant_enumeration_disease.json",
            "label_level": "diagnosis",
        },
    }

    resolved: Dict[str, Dict[str, Any]] = {}
    for key, meta in specs.items():
        subset_root = train_root / meta["dir"]
        json_path = subset_root / meta["json"]
        img_dir = subset_root / "xrays"
        if not json_path.exists():
            raise FileNotFoundError(
                f"Missing annotation file for {key}: {json_path}. "
                f"Please verify Kaggle dataset mount and file names."
            )
        if not img_dir.exists():
            raise FileNotFoundError(
                f"Missing image directory for {key}: {img_dir}. "
                f"Please verify Kaggle dataset mount and directory structure."
            )
        resolved[key] = {
            "json": json_path,
            "images": img_dir,
            "label_level": str(meta["label_level"]),
        }

    unlabeled_candidates = [
        root / "unlabelled",
        root / "training_data" / "unlabelled",
        train_root / "unlabelled",
    ]
    unlabeled_dir = next((p for p in unlabeled_candidates if p.exists()), None)
    resolved["unlabeled"] = {
        "images": unlabeled_dir,
    }
    # Backward-compatible alias for older cells that used British spelling.
    resolved["unlabelled"] = resolved["unlabeled"]

    print("Resolved subset paths:")
    for k, v in resolved.items():
        if k == "unlabelled":
            continue
        if k == "unlabeled":
            print(f"- {k}: images={v['images']}")
        else:
            print(f"- {k}: json={v['json']} images={v['images']} label_level={v['label_level']}")
    return resolved


PATHS = resolve_dentex_paths(CFG)


## 4. Dataset Classes: DentexDataset and UnlabeledDataset

Labeled data are parsed from COCO JSON. Missing supervision is represented as None for compatible partial-label training.


In [ ]:
def coco_to_xyxy(box: Sequence[float]) -> List[float]:
    """Convert COCO bbox [x, y, w, h] to [x1, y1, x2, y2]."""
    x, y, w, h = box
    return [float(x), float(y), float(x + w), float(y + h)]


def xyxy_to_coco(box: Sequence[float]) -> List[float]:
    """Convert bbox [x1, y1, x2, y2] to COCO [x, y, w, h]."""
    x1, y1, x2, y2 = box
    return [float(x1), float(y1), float(x2 - x1), float(y2 - y1)]


def safe_int(value: Any) -> Optional[int]:
    """Safely cast to int, returning None on failure."""
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def normalize_class_id(raw_value: Any, num_classes: int) -> int:
    """Normalize class IDs that may be either 0-based or 1-based."""
    value = safe_int(raw_value)
    if value is None:
        return -1
    if 1 <= value <= num_classes:
        return value
    if 0 <= value < num_classes:
        return value + 1
    return -1


def extract_fdi_from_text(text: Any) -> Tuple[int, int]:
    """Extract FDI-like two-digit code (11-48) from free text."""
    if text is None:
        return -1, -1
    match_fdi = re.search(r"\b([1-4][1-8])\b", str(text))
    if not match_fdi:
        return -1, -1
    fdi_val = int(match_fdi.group(1))
    return fdi_val // 10, fdi_val % 10


def build_id_name_map(raw_categories: Any) -> Dict[int, str]:
    """Build {category_id: category_name} from a COCO-style category list."""
    id_to_name: Dict[int, str] = {}
    if not isinstance(raw_categories, list):
        return id_to_name
    for item in raw_categories:
        if not isinstance(item, dict) or "id" not in item:
            continue
        cid = safe_int(item.get("id"))
        if cid is None:
            continue
        id_to_name[cid] = str(item.get("name", ""))
    return id_to_name


def ensure_coco_compat_schema(coco: COCO) -> None:
    """Ensure COCO has categories and annotation category_id for downstream eval."""
    dataset = coco.dataset

    annotations = dataset.get("annotations", [])
    if not isinstance(annotations, list):
        annotations = []

    had_missing_category_id = False
    derived_cat_ids: set[int] = set()
    for ann in annotations:
        if not isinstance(ann, dict):
            continue

        cat_id = safe_int(ann.get("category_id"))
        if cat_id is None:
            had_missing_category_id = True
            cat_id = safe_int(ann.get("category_id_1"))
            if cat_id is None:
                cat_id = safe_int(ann.get("category_id_2"))
            if cat_id is None:
                cat_id = 1
            ann["category_id"] = int(cat_id)

        derived_cat_ids.add(int(cat_id))

    raw_categories = dataset.get("categories", [])
    categories_ok = isinstance(raw_categories, list) and len(raw_categories) > 0

    if not categories_ok:
        synthesized = [
            {"id": int(cid), "name": str(cid), "supercategory": "tooth"}
            for cid in sorted(derived_cat_ids)
        ]
        if len(synthesized) == 0:
            synthesized = [{"id": 1, "name": "tooth", "supercategory": "tooth"}]
        dataset["categories"] = synthesized

    if had_missing_category_id or not categories_ok:
        coco.createIndex()


def parse_quadrant_enumeration(
    ann: Dict[str, Any],
    category_name: str | None = None,
    category1_name: str | None = None,
    category2_name: str | None = None,
    category3_name: str | None = None,
) -> Tuple[int, int]:
    """Parse quadrant/enumeration from standard and DENTEX non-standard fields."""
    q = normalize_class_id(
        ann.get("quadrant", ann.get("quadrant_id", ann.get("Q", ann.get("category_id_1", ann.get("category_id"))))),
        num_classes=4,
    )
    n = normalize_class_id(
        ann.get("enumeration", ann.get("tooth_number", ann.get("N", ann.get("category_id_2")))),
        num_classes=8,
    )

    if q == -1 or n == -1:
        # Try explicit FDI fields first when available.
        for raw_fdi in [ann.get("fdi"), ann.get("fdi_id")]:
            fdi_val = safe_int(raw_fdi)
            if fdi_val is None or not (11 <= fdi_val <= 48):
                continue
            fq = fdi_val // 10
            fn = fdi_val % 10
            if q == -1 and 1 <= fq <= 4:
                q = fq
            if n == -1 and 1 <= fn <= 8:
                n = fn

    if q == -1 or n == -1:
        # Fall back to names in categories/categories_1/categories_2/categories_3.
        for text in [category_name, category1_name, category2_name, category3_name]:
            fq, fn = extract_fdi_from_text(text)
            if q == -1 and fq != -1:
                q = fq
            if n == -1 and fn != -1:
                n = fn

    if q == -1 and category1_name is not None:
        q = normalize_class_id(category1_name, num_classes=4)
    if n == -1 and category2_name is not None:
        n = normalize_class_id(category2_name, num_classes=8)

    return q, n


def read_grayscale_as_rgb(image_path: Path) -> np.ndarray:
    """Read grayscale X-ray and replicate channels to RGB."""
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Unable to read image at {image_path}")
    rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    return rgb


class DentexDataset(Dataset):
    """COCO-backed DENTEX dataset supporting multiple label levels."""

    def __init__(
        self,
        annotation_json: Path,
        image_dir: Path,
        label_level: str,
        cfg: DentexConfig,
        transforms: Optional[A.Compose] = None,
        include_masks: bool = True,
    ) -> None:
        self.annotation_json = annotation_json
        self.image_dir = image_dir
        self.label_level = label_level
        self.cfg = cfg
        self.transforms = transforms
        self.include_masks = include_masks

        self.coco = COCO(str(annotation_json))
        ensure_coco_compat_schema(self.coco)
        self.image_ids = sorted(self.coco.getImgIds())

        # DENTEX mixes standard COCO categories with task-specific categories_1/2/3 keys.
        self.category_maps: Dict[str, Dict[int, str]] = {
            "categories": build_id_name_map(self.coco.dataset.get("categories", [])),
            "categories_1": build_id_name_map(self.coco.dataset.get("categories_1", [])),
            "categories_2": build_id_name_map(self.coco.dataset.get("categories_2", [])),
            "categories_3": build_id_name_map(self.coco.dataset.get("categories_3", [])),
        }
        self.category_map = self.category_maps["categories"]

    def __len__(self) -> int:
        """Return dataset size."""
        return len(self.image_ids)

    def _lookup_category_name(self, ann: Dict[str, Any], id_key: str, cat_key: str) -> str:
        """Resolve annotation category names from any available category map."""
        cid = safe_int(ann.get(id_key))
        if cid is None:
            return ""
        return self.category_maps.get(cat_key, {}).get(cid, "")

    def _extract_targets(self, image_id: int) -> Dict[str, Any]:
        """Extract box, class, and optional mask targets for one image."""
        ann_ids = self.coco.getAnnIds(imgIds=[image_id])
        anns = self.coco.loadAnns(ann_ids)

        boxes_xyxy: List[List[float]] = []
        quadrants: List[int] = []
        enumerations: List[int] = []
        masks: List[np.ndarray] = []

        for ann in anns:
            bbox = ann.get("bbox", None)
            if bbox is None or len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
                continue

            boxes_xyxy.append(coco_to_xyxy(bbox))

            cat_name = self._lookup_category_name(ann, id_key="category_id", cat_key="categories")
            cat1_name = self._lookup_category_name(ann, id_key="category_id_1", cat_key="categories_1")
            cat2_name = self._lookup_category_name(ann, id_key="category_id_2", cat_key="categories_2")
            cat3_name = self._lookup_category_name(ann, id_key="category_id_3", cat_key="categories_3")

            q, n = parse_quadrant_enumeration(
                ann,
                category_name=cat_name,
                category1_name=cat1_name,
                category2_name=cat2_name,
                category3_name=cat3_name,
            )
            quadrants.append(q)
            enumerations.append(n)

            if self.include_masks and ann.get("segmentation") is not None:
                try:
                    masks.append(self.coco.annToMask(ann).astype(np.uint8))
                except Exception:
                    # Keep training robust when segmentation polygons are malformed.
                    continue

        targets: Dict[str, Any] = {
            "boxes_xyxy": boxes_xyxy,
            "quadrants": quadrants,
            "enumerations": enumerations,
            "masks": masks,
        }
        return targets

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """Get one transformed training sample dict."""
        image_id = self.image_ids[idx]
        img_info = self.coco.loadImgs([image_id])[0]
        image_path = self.image_dir / img_info["file_name"]

        image = read_grayscale_as_rgb(image_path)
        target = self._extract_targets(image_id)

        boxes_xyxy = target["boxes_xyxy"]
        quadrants = target["quadrants"]
        enumerations = target["enumerations"]
        masks = target["masks"]

        boxes_coco = [xyxy_to_coco(b) for b in boxes_xyxy]

        if len(enumerations) == 0:
            enumerations_for_transform = [-1 for _ in quadrants]
        else:
            enumerations_for_transform = enumerations

        transform_input: Dict[str, Any] = {
            "image": image,
            "bboxes": boxes_coco,
            "quadrants": quadrants,
            "enumerations": enumerations_for_transform,
        }

        if len(masks) > 0:
            transform_input["masks"] = masks

        if self.transforms is not None:
            transformed = self.transforms(**transform_input)
            image_tensor = transformed["image"]
            t_bboxes = transformed["bboxes"]
            t_quadrants = transformed["quadrants"]
            t_enumerations = transformed["enumerations"]
            t_masks = transformed.get("masks", [])
        else:
            image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            t_bboxes = boxes_coco
            t_quadrants = quadrants
            t_enumerations = enumerations_for_transform
            t_masks = masks

        if len(t_bboxes) > 0:
            boxes_xyxy_t = torch.tensor([coco_to_xyxy(b) for b in t_bboxes], dtype=torch.float32)
        else:
            boxes_xyxy_t = torch.zeros((0, 4), dtype=torch.float32)
        quadrants_t = torch.tensor(t_quadrants, dtype=torch.long)

        enumerations_t: Optional[torch.Tensor]
        if self.label_level in {"quadrant_enumeration", "diagnosis"}:
            enumerations_t = torch.tensor(t_enumerations, dtype=torch.long)
        else:
            enumerations_t = None

        masks_t: Optional[torch.Tensor]
        if self.label_level == "diagnosis" and len(t_masks) > 0:
            masks_t = torch.stack([torch.from_numpy(np.array(m, dtype=np.uint8)) for m in t_masks]).float()
        else:
            masks_t = None

        sample: Dict[str, Any] = {
            "image": image_tensor,
            "boxes": boxes_xyxy_t,
            "quadrants": quadrants_t,
            "enumerations": enumerations_t,
            "masks": masks_t,
            "image_id": torch.tensor([image_id], dtype=torch.long),
            "label_level": self.label_level,
            "orig_size": torch.tensor([img_info["height"], img_info["width"]], dtype=torch.long),
            "path": str(image_path),
        }
        return sample


class UnlabeledDataset(Dataset):
    """Unlabeled image dataset for SimMIM pretraining."""

    def __init__(self, image_paths: Sequence[Path], transforms: Optional[A.Compose] = None) -> None:
        self.image_paths = list(image_paths)
        self.transforms = transforms

    def __len__(self) -> int:
        """Return number of unlabeled images."""
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """Return one unlabeled image sample."""
        image_path = self.image_paths[idx]
        image = read_grayscale_as_rgb(image_path)

        if self.transforms is not None:
            try:
                transformed = self.transforms(image=image)
            except ValueError as exc:
                # Some transforms include bbox label_fields; unlabeled samples must pass empty fields.
                if "label_fields" not in str(exc):
                    raise
                transformed = self.transforms(image=image, bboxes=[], quadrants=[], enumerations=[])
            image_t = transformed["image"]
        else:
            image_t = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        return {
            "image": image_t,
            "path": str(image_path),
        }


## 5. Albumentations Pipelines and Custom collate_fn

Transforms are defined with explicit no-horizontal-flip policy to preserve FDI side semantics.


In [ ]:
def build_train_transforms(cfg: DentexConfig) -> A.Compose:
    """Build training transforms for DENTEX."""
    # NOTE: HorizontalFlip is intentionally not used because it breaks FDI left/right semantics.
    return A.Compose(
        [
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
            A.Resize(cfg.img_h, cfg.img_w),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=5, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ],
        bbox_params=A.BboxParams(
            format="coco",
            label_fields=["quadrants", "enumerations"],
            min_visibility=0.0,
            clip=True,
        ),
    )


def build_val_transforms(cfg: DentexConfig) -> A.Compose:
    """Build validation transforms for DENTEX."""
    return A.Compose(
        [
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
            A.Resize(cfg.img_h, cfg.img_w),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ],
        bbox_params=A.BboxParams(
            format="coco",
            label_fields=["quadrants", "enumerations"],
            min_visibility=0.0,
            clip=True,
        ),
    )


def dentex_collate_fn(batch: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    """Collate variable-instance targets while stacking images."""
    images = torch.stack([sample["image"] for sample in batch], dim=0)
    targets = []
    label_levels = []
    for sample in batch:
        targets.append({
            "boxes": sample["boxes"],
            "quadrants": sample["quadrants"],
            "enumerations": sample["enumerations"],
            "masks": sample["masks"],
            "image_id": sample["image_id"],
            "orig_size": sample["orig_size"],
            "label_level": sample["label_level"],
            "path": sample["path"],
        })
        label_levels.append(sample["label_level"])

    return {
        "images": images,
        "targets": targets,
        "label_levels": label_levels,
    }


TRAIN_TRANSFORMS = build_train_transforms(CFG)
VAL_TRANSFORMS = build_val_transforms(CFG)


## 6. Hierarchical Batch Sampling (40/30/30)

Batches are mixed across annotation levels with deterministic sampling: diagnosis 40%, quadrant+enumeration 30%, quadrant-only 30%.


In [ ]:
def bin_stratification_key(coco: COCO, image_id: int, max_bin: int = 10) -> int:
    """Create a simple stratification key from instance count bins."""
    ann_count = len(coco.getAnnIds(imgIds=[image_id]))
    return min(max_bin, ann_count)


def split_image_ids_stratified(
    coco: COCO,
    val_fraction: float,
    seed: int,
) -> Tuple[List[int], List[int]]:
    """Create deterministic stratified split by image-level instance bins."""
    rng = np.random.default_rng(seed)
    ids = np.array(sorted(coco.getImgIds()))
    keys = np.array([bin_stratification_key(coco, img_id) for img_id in ids])

    train_ids: List[int] = []
    val_ids: List[int] = []

    for key in np.unique(keys):
        bucket = ids[keys == key].copy()
        rng.shuffle(bucket)

        # Keep at least one sample in train for tiny buckets.
        if len(bucket) == 1:
            train_ids.extend(bucket.tolist())
            continue

        n_val = max(1, int(len(bucket) * val_fraction))
        n_val = min(n_val, len(bucket) - 1)
        val_ids.extend(bucket[:n_val].tolist())
        train_ids.extend(bucket[n_val:].tolist())

    return sorted(train_ids), sorted(val_ids)


class HierarchicalBatchSampler(Sampler[List[int]]):
    """Sample mixed batches from subset index pools using fixed ratios."""

    def __init__(
        self,
        subset_indices: Dict[str, Sequence[int]],
        batch_size: int,
        num_batches: int,
        ratios: Dict[str, float],
        seed: int,
    ) -> None:
        self.subset_indices = {k: np.array(v, dtype=np.int64) for k, v in subset_indices.items()}
        self.batch_size = batch_size
        self.num_batches = num_batches
        self.ratios = ratios
        self.seed = seed

        assert abs(sum(self.ratios.values()) - 1.0) < 1e-6
        for key, indices in self.subset_indices.items():
            if len(indices) == 0:
                raise ValueError(f"Subset {key} has zero samples.")

    def __len__(self) -> int:
        """Return number of batches per epoch."""
        return self.num_batches

    def __iter__(self) -> Iterable[List[int]]:
        """Yield mixed global indices for each batch."""
        rng = np.random.default_rng(self.seed + int(time.time()))
        subset_keys = list(self.subset_indices.keys())
        probs = np.array([self.ratios[k] for k in subset_keys], dtype=np.float64)

        for _ in range(self.num_batches):
            chosen_subsets = rng.choice(subset_keys, size=self.batch_size, replace=True, p=probs)
            batch_indices: List[int] = []
            for subset_name in chosen_subsets:
                subset_pool = self.subset_indices[subset_name]
                idx = int(rng.choice(subset_pool, size=1, replace=True)[0])
                batch_indices.append(idx)

            # Runtime sanity checks for label coverage in mixed batches.
            assert len(batch_indices) == self.batch_size
            yield batch_indices


def build_datasets_and_loaders(cfg: DentexConfig) -> Dict[str, Any]:
    """Construct train/val datasets and hierarchical train loader."""
    train_datasets: Dict[str, DentexDataset] = {}
    val_datasets: Dict[str, DentexDataset] = {}

    for subset_key in ["quadrant", "quadrant_enumeration", "quadrant_enumeration_disease"]:
        entry = PATHS[subset_key]
        label_level = str(entry["label_level"])

        base_ds = DentexDataset(
            annotation_json=entry["json"],
            image_dir=entry["images"],
            label_level=label_level,
            cfg=cfg,
            transforms=None,
            include_masks=True,
        )
        train_ids, val_ids = split_image_ids_stratified(base_ds.coco, val_fraction=0.2, seed=cfg.seed)

        train_ds = DentexDataset(
            annotation_json=entry["json"],
            image_dir=entry["images"],
            label_level=label_level,
            cfg=cfg,
            transforms=TRAIN_TRANSFORMS,
            include_masks=True,
        )
        val_ds = DentexDataset(
            annotation_json=entry["json"],
            image_dir=entry["images"],
            label_level=label_level,
            cfg=cfg,
            transforms=VAL_TRANSFORMS,
            include_masks=True,
        )

        train_ds.image_ids = train_ids
        val_ds.image_ids = val_ids

        train_datasets[subset_key] = train_ds
        val_datasets[subset_key] = val_ds

    concat_train = ConcatDataset([
        train_datasets["quadrant_enumeration_disease"],
        train_datasets["quadrant_enumeration"],
        train_datasets["quadrant"],
    ])

    lengths = [
        len(train_datasets["quadrant_enumeration_disease"]),
        len(train_datasets["quadrant_enumeration"]),
        len(train_datasets["quadrant"]),
    ]
    offsets = np.cumsum([0] + lengths)

    subset_indices = {
        "diagnosis": np.arange(offsets[0], offsets[1]).tolist(),
        "quadrant_enumeration": np.arange(offsets[1], offsets[2]).tolist(),
        "quadrant": np.arange(offsets[2], offsets[3]).tolist(),
    }

    sampler = HierarchicalBatchSampler(
        subset_indices=subset_indices,
        batch_size=cfg.train_batch_size,
        num_batches=max(1, cfg.train_iterations // max(1, cfg.train_batch_size)),
        ratios={
            "diagnosis": cfg.ratio_diagnosis,
            "quadrant_enumeration": cfg.ratio_quad_enum,
            "quadrant": cfg.ratio_quad_only,
        },
        seed=cfg.seed,
    )

    train_loader = DataLoader(
        concat_train,
        batch_sampler=sampler,
        collate_fn=dentex_collate_fn,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        persistent_workers=cfg.persistent_workers,
    )

    val_loaders = {
        key: DataLoader(
            ds,
            batch_size=cfg.train_batch_size,
            shuffle=False,
            collate_fn=dentex_collate_fn,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
            persistent_workers=cfg.persistent_workers,
        )
        for key, ds in val_datasets.items()
    }

    unlabeled_paths: List[Path] = []
    unlabeled_entry = PATHS.get("unlabeled", PATHS.get("unlabelled", {}))
    unlabeled_root = unlabeled_entry.get("images") if isinstance(unlabeled_entry, dict) else None
    if unlabeled_root is not None:
        unlabeled_paths = sorted(
            [
                p
                for p in Path(unlabeled_root).rglob("*")
                if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
            ]
        )

    unlabeled_ds = UnlabeledDataset(image_paths=unlabeled_paths, transforms=VAL_TRANSFORMS)
    unlabeled_loader = DataLoader(
        unlabeled_ds,
        batch_size=cfg.simmim_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    ) if len(unlabeled_ds) > 0 else None

    return {
        "train_loader": train_loader,
        "val_loaders": val_loaders,
        "unlabeled_loader": unlabeled_loader,
        "train_datasets": train_datasets,
        "val_datasets": val_datasets,
    }


DATA_BUNDLES = build_datasets_and_loaders(CFG)
print("Train loader and validation loaders are ready.")
print(f"Unlabeled loader available: {DATA_BUNDLES['unlabeled_loader'] is not None}")


## 7. Core Shared Blocks: CoordConv2d and SwinFPN

The Swin encoder and FPN are shared globally by all heads.


In [ ]:
class CoordConv2d(nn.Module):
    """Conv2d with appended normalized X/Y coordinate channels."""

    def __init__(self, in_channels: int, out_channels: int, **kwargs: Any) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels + 2, out_channels, **kwargs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with coordinate augmentation."""
        bsz, _, hgt, wdt = x.shape
        y_coords = torch.linspace(-1.0, 1.0, hgt, device=x.device, dtype=x.dtype).view(1, 1, hgt, 1).expand(bsz, 1, hgt, wdt)
        x_coords = torch.linspace(-1.0, 1.0, wdt, device=x.device, dtype=x.dtype).view(1, 1, 1, wdt).expand(bsz, 1, hgt, wdt)
        x_aug = torch.cat([x, y_coords, x_coords], dim=1)
        return self.conv(x_aug)


class SwinFPN(nn.Module):
    """Shared Swin backbone and top-down FPN."""

    def __init__(self, cfg: DentexConfig) -> None:
        super().__init__()
        self.cfg = cfg

        self.swin = timm.create_model(
            cfg.swin_model,
            pretrained=cfg.swin_pretrained,
            features_only=True,
            out_indices=cfg.fpn_out_indices,
            img_size=(cfg.img_h, cfg.img_w),
        )

        swin_channels = self.swin.feature_info.channels()
        assert len(swin_channels) == len(cfg.fpn_levels), "Expected 4 backbone feature levels for P2-P5."

        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(ch, cfg.fpn_out_channels, kernel_size=1)
            for ch in swin_channels
        ])

        self.fpn_convs = nn.ModuleList([
            nn.Conv2d(cfg.fpn_out_channels, cfg.fpn_out_channels, kernel_size=3, padding=1)
            for _ in swin_channels
        ])

    def _to_nchw_if_needed(self, feat: torch.Tensor, expected_channels: int) -> torch.Tensor:
        """Convert backbone feature to NCHW when model emits NHWC tensors."""
        if feat.ndim != 4:
            raise RuntimeError(f"Backbone feature must be 4D, got shape {tuple(feat.shape)}")

        if feat.shape[1] == expected_channels:
            return feat

        if feat.shape[-1] == expected_channels:
            return feat.permute(0, 3, 1, 2).contiguous()

        raise RuntimeError(
            f"Unable to align feature tensor with expected channels {expected_channels}. "
            f"Received shape {tuple(feat.shape)}."
        )

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        """Return multiscale FPN features [P2, P3, P4, P5]."""
        raw_feats = self.swin(x)
        assert len(raw_feats) == len(self.cfg.fpn_levels)

        feats = [
            self._to_nchw_if_needed(feat, expected_channels=self.lateral_convs[idx].in_channels)
            for idx, feat in enumerate(raw_feats)
        ]

        laterals = [layer(f) for layer, f in zip(self.lateral_convs, feats)]

        for level in range(len(laterals) - 1, 0, -1):
            laterals[level - 1] = laterals[level - 1] + F.interpolate(
                laterals[level],
                size=laterals[level - 1].shape[-2:],
                mode="nearest",
            )

        outs = [conv(lat) for conv, lat in zip(self.fpn_convs, laterals)]

        bsz = x.shape[0]
        for feat in outs:
            assert feat.shape[0] == bsz, f"Batch mismatch in FPN output: expected {bsz}, got {feat.shape[0]}"
            assert feat.shape[1] == self.cfg.fpn_out_channels, (
                f"Expected FPN channels {self.cfg.fpn_out_channels}, got {feat.shape[1]}"
            )

        return outs


## 8. Quadrant Detection Head and FPN ROIAlign Cropping

Quadrant detection is the stage-1 divide-and-conquer block that drives per-quadrant feature cropping.


In [ ]:
class QuadrantDetectionHead(nn.Module):
    """Predict 4 coarse quadrant boxes and confidence logits from FPN P4/P5."""

    def __init__(self, cfg: DentexConfig) -> None:
        super().__init__()
        self.cfg = cfg
        in_ch = cfg.fpn_out_channels
        hidden = cfg.fpn_out_channels

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.box_head = nn.Linear(hidden, cfg.num_quadrants * 4)
        self.cls_head = nn.Linear(hidden, cfg.num_quadrants)

    def forward(self, fpn_feats: List[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return normalized quadrant boxes [B,4,4] and logits [B,4]."""
        feat = fpn_feats[2]  # P4 by design
        bsz = feat.shape[0]
        x = self.stem(feat)
        pooled = self.pool(x).view(bsz, -1)

        boxes = self.box_head(pooled).view(bsz, self.cfg.num_quadrants, 4).sigmoid()
        scores = self.cls_head(pooled)

        assert boxes.shape == (bsz, self.cfg.num_quadrants, 4), (
            f"Expected {(bsz, self.cfg.num_quadrants, 4)}, got {tuple(boxes.shape)}"
        )
        assert scores.shape == (bsz, self.cfg.num_quadrants), (
            f"Expected {(bsz, self.cfg.num_quadrants)}, got {tuple(scores.shape)}"
        )
        return boxes, scores


def cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    """Convert [cx,cy,w,h] in normalized space to [x1,y1,x2,y2]."""
    cx, cy, w, h = boxes.unbind(dim=-1)
    x1 = (cx - 0.5 * w).clamp(0.0, 1.0)
    y1 = (cy - 0.5 * h).clamp(0.0, 1.0)
    x2 = (cx + 0.5 * w).clamp(0.0, 1.0)
    y2 = (cy + 0.5 * h).clamp(0.0, 1.0)
    return torch.stack([x1, y1, x2, y2], dim=-1)


def giou_loss_xyxy(pred_xyxy: torch.Tensor, tgt_xyxy: torch.Tensor) -> torch.Tensor:
    """Compute mean GIoU loss for aligned box tensors."""
    if pred_xyxy.numel() == 0 or tgt_xyxy.numel() == 0:
        return torch.tensor(0.0, device=pred_xyxy.device if pred_xyxy.numel() else tgt_xyxy.device)
    giou = generalized_box_iou(pred_xyxy, tgt_xyxy)
    diag = torch.diag(giou)
    return (1.0 - diag).mean()


def normalized_xyxy_to_absolute_xyxy(boxes: torch.Tensor, height: int, width: int) -> torch.Tensor:
    """Map normalized boxes to absolute pixel coordinates."""
    scale = torch.tensor([width, height, width, height], dtype=boxes.dtype, device=boxes.device)
    return boxes * scale


def roi_align_fpn(
    fpn_feats: List[torch.Tensor],
    quad_boxes_xyxy_norm: torch.Tensor,
    cfg: DentexConfig,
) -> List[List[torch.Tensor]]:
    """Crop all FPN levels into per-quadrant feature pyramids using ROIAlign."""
    bsz = quad_boxes_xyxy_norm.shape[0]
    assert quad_boxes_xyxy_norm.shape[1] == cfg.num_quadrants

    quad_crops: List[List[torch.Tensor]] = []
    for q_idx in range(cfg.num_quadrants):
        per_level_crops: List[torch.Tensor] = []

        for feat in fpn_feats:
            _, _, hgt, wdt = feat.shape
            rois: List[torch.Tensor] = []
            for b in range(bsz):
                box = quad_boxes_xyxy_norm[b, q_idx]
                box_abs = normalized_xyxy_to_absolute_xyxy(box, height=hgt, width=wdt)
                roi = torch.cat([torch.tensor([float(b)], device=feat.device, dtype=feat.dtype), box_abs])
                rois.append(roi)
            rois_t = torch.stack(rois, dim=0)

            out_size = (
                max(4, hgt // 2),
                max(4, wdt // 2),
            )
            crop = roi_align(
                input=feat,
                boxes=rois_t,
                output_size=out_size,
                spatial_scale=1.0,
                sampling_ratio=-1,
                aligned=True,
            )
            per_level_crops.append(crop)

        quad_crops.append(per_level_crops)

    return quad_crops


## 9. Stage-2 Parallel Heads: Shared U-Net and DINO

A single shared U-Net head and a DINO detection head are reused across all quadrant crops.


In [ ]:
class DoubleConv(nn.Module):
    """Two-layer Conv-BN-ReLU block."""

    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward block."""
        return self.block(x)


class UNetSegmentationHead(nn.Module):
    """U-Net-like decoder over cropped FPN features."""

    def __init__(self, cfg: DentexConfig) -> None:
        super().__init__()
        ch1, ch2, ch3, ch4 = cfg.seg_unet_channels
        in_ch = cfg.fpn_out_channels

        self.dec5_4 = DoubleConv(in_ch + in_ch, ch1)
        self.dec4_3 = DoubleConv(ch1 + in_ch, ch2)
        self.dec3_2 = DoubleConv(ch2 + in_ch, ch3)
        self.dec2_1 = DoubleConv(ch3, ch4)
        self.head = nn.Conv2d(ch4, cfg.seg_num_classes, kernel_size=1)

    def forward(self, q_feats: List[torch.Tensor]) -> torch.Tensor:
        """Return segmentation logits for one quadrant crop batch."""
        p2, p3, p4, p5 = q_feats

        x = F.interpolate(p5, size=p4.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, p4], dim=1)
        x = self.dec5_4(x)

        x = F.interpolate(x, size=p3.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, p3], dim=1)
        x = self.dec4_3(x)

        x = F.interpolate(x, size=p2.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, p2], dim=1)
        x = self.dec3_2(x)

        x = F.interpolate(x, scale_factor=2.0, mode="bilinear", align_corners=False)
        x = self.dec2_1(x)
        logits = self.head(x)

        assert logits.ndim == 4, f"Expected 4D segmentation logits, got {logits.shape}"
        return logits


class DINODetectionHead(nn.Module):
    """DINO-style detection head with optional CoordConv projection."""

    def __init__(self, cfg: DentexConfig) -> None:
        super().__init__()
        self.cfg = cfg
        hidden = cfg.dino_hidden_dim

        proj_conv_cls = CoordConv2d if cfg.use_coordconv else nn.Conv2d
        # NOTE: CoordConv is used to expose explicit spatial coordinates for FDI assignment.
        self.input_proj = proj_conv_cls(cfg.fpn_out_channels, hidden, kernel_size=1)
        self.encoder_proj = proj_conv_cls(hidden, hidden, kernel_size=3, padding=1)

        self.query_embed = nn.Embedding(cfg.dino_num_queries, hidden)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden,
            nhead=cfg.dino_num_heads,
            dim_feedforward=cfg.dino_ffn_dim,
            dropout=cfg.dino_dropout,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=cfg.dino_num_decoder_layers)

        self.bbox_head = nn.Linear(hidden, 4)
        self.class_head = nn.Linear(hidden, cfg.dino_num_output_classes)

    def forward(self, q_feats: List[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return query box predictions and class logits."""
        feat = q_feats[2]  # P4 crop
        feat = self.input_proj(feat)
        feat = self.encoder_proj(feat)

        bsz, _, _, _ = feat.shape
        memory = feat.flatten(2).permute(0, 2, 1)

        queries = self.query_embed.weight.unsqueeze(0).expand(bsz, -1, -1)
        dec_out = self.decoder(tgt=queries, memory=memory)

        boxes = self.bbox_head(dec_out).sigmoid()
        classes = self.class_head(dec_out)

        assert boxes.shape == (bsz, self.cfg.dino_num_queries, 4), (
            f"Expected {(bsz, self.cfg.dino_num_queries, 4)}, got {tuple(boxes.shape)}"
        )
        assert classes.shape == (bsz, self.cfg.dino_num_queries, self.cfg.dino_num_output_classes), (
            f"Expected {(bsz, self.cfg.dino_num_queries, self.cfg.dino_num_output_classes)}, got {tuple(classes.shape)}"
        )
        return boxes, classes


## 10. Full DentexPipeline Graph Assembly

This section assembles the full model graph with shared features and per-quadrant parallel stage-2 heads.


In [ ]:
class PathologyHeadPlaceholder(nn.Module):
    """Placeholder for future pathology module (out of scope in this notebook)."""

    def __init__(self) -> None:
        super().__init__()

    def forward(self, *args: Any, **kwargs: Any) -> Any:
        """Forward placeholder."""
        _ = (args, kwargs)
        # TODO: Implement pathology detection module in a later phase.
        return None


class DentexPipeline(nn.Module):
    """End-to-end DENTEX model with shared backbone and two stage-2 branches."""

    def __init__(self, cfg: DentexConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.backbone_fpn = SwinFPN(cfg)
        self.quad_head = QuadrantDetectionHead(cfg)
        self.seg_head = UNetSegmentationHead(cfg)
        self.dino_head = DINODetectionHead(cfg)
        self.pathology_head = PathologyHeadPlaceholder()

    def forward(
        self,
        images: torch.Tensor,
        targets: Optional[List[Dict[str, Any]]] = None,
    ) -> Dict[str, Any]:
        """Forward pass for training or inference."""
        fpn_feats = self.backbone_fpn(images)

        quad_boxes_cxcywh, quad_logits = self.quad_head(fpn_feats)
        quad_boxes_xyxy = cxcywh_to_xyxy(quad_boxes_cxcywh)

        quad_crops = roi_align_fpn(fpn_feats, quad_boxes_xyxy, self.cfg)

        seg_logits_by_quad: List[torch.Tensor] = []
        dino_boxes_by_quad: List[torch.Tensor] = []
        dino_logits_by_quad: List[torch.Tensor] = []

        for q_feats in quad_crops:
            seg_logits = self.seg_head(q_feats)
            dino_boxes, dino_logits = self.dino_head(q_feats)
            seg_logits_by_quad.append(seg_logits)
            dino_boxes_by_quad.append(dino_boxes)
            dino_logits_by_quad.append(dino_logits)

        out: Dict[str, Any] = {
            "quad_boxes_cxcywh": quad_boxes_cxcywh,
            "quad_boxes_xyxy": quad_boxes_xyxy,
            "quad_logits": quad_logits,
            "seg_logits_by_quad": seg_logits_by_quad,
            "dino_boxes_by_quad": dino_boxes_by_quad,
            "dino_logits_by_quad": dino_logits_by_quad,
        }

        if self.training and targets is not None:
            return out

        quad_scores = torch.sigmoid(quad_logits)
        out["quad_scores"] = quad_scores
        return out


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = DentexPipeline(CFG).to(DEVICE)
print(f"Model initialized on {DEVICE}.")


## 11. Hungarian Matching and Multi-Task Loss Aggregation

This section implements DETR-style matching/losses, segmentation losses, and label-level masking for partial supervision.


In [ ]:
def safe_sigmoid(x: torch.Tensor, eps: float) -> torch.Tensor:
    """Numerically safe sigmoid output clipping."""
    return torch.sigmoid(x).clamp(eps, 1.0 - eps)


def dice_loss_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float) -> torch.Tensor:
    """Compute Dice loss for binary segmentation logits."""
    probs = safe_sigmoid(logits, eps)
    intersection = (probs * targets).sum(dim=(1, 2, 3))
    union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()


def build_quadrant_gt_boxes(target: Dict[str, Any], cfg: DentexConfig) -> torch.Tensor:
    """Derive 4 GT quadrant boxes from per-tooth boxes and quadrant IDs."""
    boxes = target["boxes"]
    quadrants = target["quadrants"]

    default_boxes = torch.tensor(
        [
            [0.5, 0.0, 1.0, 0.5],
            [0.0, 0.0, 0.5, 0.5],
            [0.0, 0.5, 0.5, 1.0],
            [0.5, 0.5, 1.0, 1.0],
        ],
        dtype=torch.float32,
        device=boxes.device if boxes.numel() else DEVICE,
    )

    if boxes.numel() == 0:
        return default_boxes

    norm_scale = torch.tensor([cfg.img_w, cfg.img_h, cfg.img_w, cfg.img_h], dtype=boxes.dtype, device=boxes.device)
    boxes_n = boxes / norm_scale

    gt_quads = []
    for q in range(1, cfg.num_quadrants + 1):
        idx = torch.where(quadrants == q)[0]
        if len(idx) == 0:
            gt_quads.append(default_boxes[q - 1])
            continue
        q_boxes = boxes_n[idx]
        x1 = q_boxes[:, 0].min()
        y1 = q_boxes[:, 1].min()
        x2 = q_boxes[:, 2].max()
        y2 = q_boxes[:, 3].max()
        gt_quads.append(torch.stack([x1, y1, x2, y2]))

    return torch.stack(gt_quads, dim=0)


def project_boxes_into_quadrant(
    boxes_xyxy_abs: torch.Tensor,
    quad_box_xyxy_norm: torch.Tensor,
    cfg: DentexConfig,
) -> torch.Tensor:
    """Project global absolute boxes to crop-normalized [0,1] coordinates."""
    if boxes_xyxy_abs.numel() == 0:
        return boxes_xyxy_abs.new_zeros((0, 4))

    scale = torch.tensor([cfg.img_w, cfg.img_h, cfg.img_w, cfg.img_h], dtype=boxes_xyxy_abs.dtype, device=boxes_xyxy_abs.device)
    global_n = boxes_xyxy_abs / scale

    qx1, qy1, qx2, qy2 = quad_box_xyxy_norm
    qw = max(float(qx2 - qx1), cfg.eps)
    qh = max(float(qy2 - qy1), cfg.eps)

    x1 = ((global_n[:, 0] - qx1) / qw).clamp(0.0, 1.0)
    y1 = ((global_n[:, 1] - qy1) / qh).clamp(0.0, 1.0)
    x2 = ((global_n[:, 2] - qx1) / qw).clamp(0.0, 1.0)
    y2 = ((global_n[:, 3] - qy1) / qh).clamp(0.0, 1.0)
    return torch.stack([x1, y1, x2, y2], dim=-1)


def xyxy_to_cxcywh_torch(boxes: torch.Tensor) -> torch.Tensor:
    """Convert tensor boxes [x1,y1,x2,y2] to [cx,cy,w,h]."""
    x1, y1, x2, y2 = boxes.unbind(dim=-1)
    cx = (x1 + x2) * 0.5
    cy = (y1 + y2) * 0.5
    w = (x2 - x1).clamp(min=0.0)
    h = (y2 - y1).clamp(min=0.0)
    return torch.stack([cx, cy, w, h], dim=-1)


class HungarianMatcher:
    """Hungarian bipartite matcher for DETR/DINO losses."""

    def __init__(self, cfg: DentexConfig) -> None:
        self.cfg = cfg

    @torch.no_grad()
    def match(
        self,
        pred_boxes: torch.Tensor,
        pred_logits: torch.Tensor,
        tgt_boxes: torch.Tensor,
        tgt_cls: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return optimal prediction and target indices."""
        if tgt_boxes.numel() == 0:
            empty = torch.zeros((0,), dtype=torch.long, device=pred_boxes.device)
            return empty, empty

        cost_l1 = torch.cdist(pred_boxes, tgt_boxes, p=1)

        pred_xyxy = cxcywh_to_xyxy(pred_boxes)
        giou = generalized_box_iou(pred_xyxy, tgt_boxes)
        cost_giou = 1.0 - giou

        probs = pred_logits.softmax(dim=-1)
        cost_cls = -probs[:, tgt_cls]

        cost = (
            self.cfg.dino_lambda_box * cost_l1
            + self.cfg.dino_lambda_giou * cost_giou
            + self.cfg.dino_lambda_cls * cost_cls
        )

        row_idx, col_idx = linear_sum_assignment(cost.detach().cpu().numpy())
        pred_idx = torch.tensor(row_idx, dtype=torch.long, device=pred_boxes.device)
        tgt_idx = torch.tensor(col_idx, dtype=torch.long, device=pred_boxes.device)
        return pred_idx, tgt_idx


class DentexMultiTaskLoss(nn.Module):
    """Compute quadrant, segmentation, and DINO losses with label masking."""

    def __init__(self, cfg: DentexConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.matcher = HungarianMatcher(cfg)
        self.bce_logits = nn.BCEWithLogitsLoss()

    def compute_quadrant_loss(
        self,
        quad_boxes_xyxy: torch.Tensor,
        quad_logits: torch.Tensor,
        targets: List[Dict[str, Any]],
    ) -> torch.Tensor:
        """Quadrant loss: box regression + BCE presence."""
        bsz = quad_boxes_xyxy.shape[0]
        gt_boxes = torch.stack([build_quadrant_gt_boxes(t, self.cfg).to(quad_boxes_xyxy.device) for t in targets], dim=0)

        box_l1 = F.l1_loss(quad_boxes_xyxy, gt_boxes)
        box_giou_vals = []
        for b in range(bsz):
            box_giou_vals.append(giou_loss_xyxy(quad_boxes_xyxy[b], gt_boxes[b]))
        box_giou = torch.stack(box_giou_vals).mean() if len(box_giou_vals) > 0 else quad_boxes_xyxy.sum() * 0.0

        gt_presence = torch.ones_like(quad_logits)
        cls_loss = self.bce_logits(quad_logits, gt_presence)

        return self.cfg.quad_loss_weight_box * (box_l1 + box_giou) + self.cfg.quad_loss_weight_cls * cls_loss

    def compute_seg_loss(
        self,
        seg_logits_by_quad: List[torch.Tensor],
        quad_boxes_xyxy: torch.Tensor,
        targets: List[Dict[str, Any]],
    ) -> torch.Tensor:
        """Segmentation loss on diagnosis samples only."""
        total_loss = torch.tensor(0.0, device=quad_boxes_xyxy.device)
        valid_count = 0

        for b, target in enumerate(targets):
            if target["label_level"] != "diagnosis" or target["masks"] is None:
                continue

            masks = target["masks"].to(quad_boxes_xyxy.device)
            quadrants = target["quadrants"].to(quad_boxes_xyxy.device)

            for q_idx in range(self.cfg.num_quadrants):
                pred_logits = seg_logits_by_quad[q_idx][b : b + 1]
                q_label = q_idx + 1
                q_mask_idx = torch.where(quadrants == q_label)[0]
                if len(q_mask_idx) == 0:
                    target_mask_full = torch.zeros((1, 1, self.cfg.img_h, self.cfg.img_w), device=pred_logits.device)
                else:
                    target_mask_full = masks[q_mask_idx].sum(dim=0, keepdim=True).clamp(0.0, 1.0).unsqueeze(0)

                q_box = quad_boxes_xyxy[b, q_idx]
                x1 = int(float(q_box[0]) * self.cfg.img_w)
                y1 = int(float(q_box[1]) * self.cfg.img_h)
                x2 = int(float(q_box[2]) * self.cfg.img_w)
                y2 = int(float(q_box[3]) * self.cfg.img_h)
                x2 = max(x2, x1 + 1)
                y2 = max(y2, y1 + 1)

                crop = target_mask_full[:, :, y1:y2, x1:x2]
                crop = F.interpolate(crop, size=pred_logits.shape[-2:], mode="nearest")

                bce = F.binary_cross_entropy_with_logits(pred_logits, crop)
                dice = dice_loss_from_logits(pred_logits, crop, self.cfg.eps)
                seg_loss = self.cfg.seg_bce_weight * bce + self.cfg.seg_dice_weight * dice

                total_loss = total_loss + seg_loss
                valid_count += 1

        if valid_count == 0:
            return total_loss
        return total_loss / valid_count

    def compute_dino_loss(
        self,
        dino_boxes_by_quad: List[torch.Tensor],
        dino_logits_by_quad: List[torch.Tensor],
        quad_boxes_xyxy: torch.Tensor,
        targets: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        """Compute DINO losses with Hungarian matching and label masking."""
        l1_total = torch.tensor(0.0, device=quad_boxes_xyxy.device)
        giou_total = torch.tensor(0.0, device=quad_boxes_xyxy.device)
        cls_total = torch.tensor(0.0, device=quad_boxes_xyxy.device)
        match_count = 0

        for b, target in enumerate(targets):
            level = target["label_level"]
            gt_boxes_abs = target["boxes"].to(quad_boxes_xyxy.device)
            gt_quadrants = target["quadrants"].to(quad_boxes_xyxy.device)
            gt_enum = target["enumerations"]
            if gt_enum is not None:
                gt_enum = gt_enum.to(quad_boxes_xyxy.device)

            for q_idx in range(self.cfg.num_quadrants):
                pred_boxes = dino_boxes_by_quad[q_idx][b]
                pred_logits = dino_logits_by_quad[q_idx][b]
                quad_box = quad_boxes_xyxy[b, q_idx]
                q_label = q_idx + 1

                gt_idx = torch.where(gt_quadrants == q_label)[0]
                gt_boxes_q = gt_boxes_abs[gt_idx]
                gt_boxes_q_local_xyxy = project_boxes_into_quadrant(gt_boxes_q, quad_box, self.cfg)
                gt_boxes_q_local = xyxy_to_cxcywh_torch(gt_boxes_q_local_xyxy)

                if gt_enum is None:
                    gt_cls_q = torch.full((len(gt_idx),), self.cfg.num_positions_per_quadrant, dtype=torch.long, device=pred_boxes.device)
                else:
                    gt_pos = gt_enum[gt_idx] - 1
                    gt_pos = gt_pos.clamp(0, self.cfg.num_positions_per_quadrant - 1)
                    gt_cls_q = gt_pos.long()

                pred_idx, tgt_idx = self.matcher.match(pred_boxes, pred_logits, gt_boxes_q_local, gt_cls_q)
                if len(pred_idx) == 0:
                    continue

                m_pred_boxes = pred_boxes[pred_idx]
                m_tgt_boxes = gt_boxes_q_local[tgt_idx]
                l1_total = l1_total + F.l1_loss(m_pred_boxes, m_tgt_boxes)

                m_pred_xyxy = cxcywh_to_xyxy(m_pred_boxes)
                m_tgt_xyxy = cxcywh_to_xyxy(m_tgt_boxes)
                giou_total = giou_total + giou_loss_xyxy(m_pred_xyxy, m_tgt_xyxy)

                if level in {"quadrant_enumeration", "diagnosis"}:
                    cls_target = gt_cls_q[tgt_idx]
                    cls_total = cls_total + F.cross_entropy(pred_logits[pred_idx], cls_target)
                else:
                    # NOTE: For quadrant-only images, classification gradients are intentionally masked.
                    cls_total = cls_total + (pred_logits[pred_idx].sum() * 0.0)

                match_count += 1

        denom = max(1, match_count)
        return {
            "l1": l1_total / denom,
            "giou": giou_total / denom,
            "cls": cls_total / denom,
        }

    def forward(self, outputs: Dict[str, Any], targets: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Compute full weighted multi-task loss dictionary."""
        quad_loss = self.compute_quadrant_loss(outputs["quad_boxes_xyxy"], outputs["quad_logits"], targets)
        seg_loss = self.compute_seg_loss(outputs["seg_logits_by_quad"], outputs["quad_boxes_xyxy"], targets)
        dino_parts = self.compute_dino_loss(
            outputs["dino_boxes_by_quad"],
            outputs["dino_logits_by_quad"],
            outputs["quad_boxes_xyxy"],
            targets,
        )
        dino_loss = (
            self.cfg.dino_lambda_box * dino_parts["l1"]
            + self.cfg.dino_lambda_giou * dino_parts["giou"]
            + self.cfg.dino_lambda_cls * dino_parts["cls"]
        )

        total = quad_loss + seg_loss + dino_loss
        return {
            "loss_total": total,
            "loss_quadrant": quad_loss,
            "loss_seg": seg_loss,
            "loss_dino": dino_loss,
            "loss_dino_l1": dino_parts["l1"],
            "loss_dino_giou": dino_parts["giou"],
            "loss_dino_cls": dino_parts["cls"],
        }


LOSS_FN = DentexMultiTaskLoss(CFG)


## 12. SimMIM Pretraining Components and Loop

This phase adapts the shared Swin backbone to dental X-rays using unlabeled images and masked reconstruction.


In [ ]:
def generate_patch_mask(
    bsz: int,
    img_h: int,
    img_w: int,
    patch_size: int,
    mask_ratio: float,
    device: torch.device,
) -> torch.Tensor:
    """Generate binary patch mask expanded to pixel resolution."""
    ph = img_h // patch_size
    pw = img_w // patch_size
    total_patches = ph * pw
    num_masked = int(total_patches * mask_ratio)

    mask = torch.zeros((bsz, total_patches), device=device)
    for i in range(bsz):
        idx = torch.randperm(total_patches, device=device)[:num_masked]
        mask[i, idx] = 1.0

    mask = mask.view(bsz, 1, ph, pw)
    mask = F.interpolate(mask, size=(img_h, img_w), mode="nearest")
    return mask


class SimMIMReconstructionHead(nn.Module):
    """Lightweight reconstruction head from P2 features to RGB image."""

    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(in_ch, out_ch, kernel_size=1),
        )

    def forward(self, x: torch.Tensor, out_hw: Tuple[int, int]) -> torch.Tensor:
        """Reconstruct image resolution output."""
        x = self.net(x)
        x = F.interpolate(x, size=out_hw, mode="bilinear", align_corners=False)
        return x


class SimMIMModel(nn.Module):
    """SimMIM pretraining model wrapping shared backbone."""

    def __init__(self, backbone_fpn: SwinFPN, cfg: DentexConfig) -> None:
        super().__init__()
        self.backbone_fpn = backbone_fpn
        self.cfg = cfg
        self.mask_token = nn.Parameter(torch.zeros(1, cfg.image_channels, 1, 1))
        nn.init.normal_(self.mask_token, std=0.02)
        self.recon_head = SimMIMReconstructionHead(cfg.fpn_out_channels, cfg.image_channels)

    def forward(self, images: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return reconstruction and mask used for masked objective."""
        bsz, _, hgt, wdt = images.shape
        mask = generate_patch_mask(
            bsz=bsz,
            img_h=hgt,
            img_w=wdt,
            patch_size=self.cfg.simmim_patch_size,
            mask_ratio=self.cfg.simmim_mask_ratio,
            device=images.device,
        )

        masked_images = images * (1.0 - mask) + self.mask_token * mask
        feats = self.backbone_fpn(masked_images)
        recon = self.recon_head(feats[0], out_hw=(hgt, wdt))
        return recon, mask


def masked_l1_loss(recon: torch.Tensor, target: torch.Tensor, mask: torch.Tensor, eps: float) -> torch.Tensor:
    """Masked-only L1 reconstruction loss."""
    diff = (recon - target).abs() * mask
    denom = mask.sum() * target.shape[1] + eps
    return diff.sum() / denom


def maybe_log_wandb(data: Dict[str, Any], step: int | None = None) -> None:
    """Log to W&B only if a run exists."""
    if wandb.run is not None:
        wandb.log(data, step=step)


def pretrain_simmim(
    model: DentexPipeline,
    unlabeled_loader: Optional[DataLoader],
    cfg: DentexConfig,
    device: torch.device,
) -> Optional[Path]:
    """Run SimMIM pretraining and return best checkpoint path."""
    if unlabeled_loader is None:
        print("No unlabeled loader found. Skipping SimMIM pretraining.")
        return None

    simmim = SimMIMModel(model.backbone_fpn, cfg).to(device)
    optimizer = torch.optim.AdamW(simmim.parameters(), lr=cfg.simmim_lr, weight_decay=cfg.simmim_weight_decay)
    amp_enabled = cfg.amp and device.type == "cuda"
    scaler = GradScaler(device=device.type, enabled=amp_enabled)

    best_loss = float("inf")
    best_path = Path(cfg.output_dir) / "simmim_backbone_best.pt"

    simmim.train()
    for epoch in range(cfg.simmim_epochs):
        epoch_losses: List[float] = []
        pbar = tqdm(unlabeled_loader, desc=f"SimMIM Epoch {epoch + 1}/{cfg.simmim_epochs}", leave=False)
        for batch in pbar:
            images = batch["image"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type=device.type, enabled=amp_enabled):
                recon, mask = simmim(images)
                loss = masked_l1_loss(recon, images, mask, cfg.eps)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_val = float(loss.detach().cpu().item())
            epoch_losses.append(loss_val)
            pbar.set_postfix({"recon_l1": f"{loss_val:.4f}"})

        epoch_loss = float(np.mean(epoch_losses)) if len(epoch_losses) > 0 else float("inf")
        maybe_log_wandb({"simmim/epoch": epoch, "simmim/recon_l1": epoch_loss})

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": simmim.backbone_fpn.state_dict(),
                    "best_loss": best_loss,
                    "cfg": asdict(cfg),
                },
                best_path,
            )
            if wandb.run is not None:
                artifact = wandb.Artifact("simmim-backbone", type="model")
                artifact.add_file(str(best_path))
                wandb.log_artifact(artifact)
                wandb.save(str(best_path))

    print(f"SimMIM best reconstruction loss: {best_loss:.6f}")
    return best_path


## 13. Main Supervised Training Loop

This section implements mixed-precision hierarchical training, scheduler warmup+cosine decay, periodic validation, and robust checkpoint persistence.


In [ ]:
def build_optimizer(cfg: DentexConfig, model: nn.Module) -> torch.optim.Optimizer:
    """Build optimizer according to configuration."""
    if cfg.optimizer.lower() == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    raise ValueError(f"Unsupported optimizer: {cfg.optimizer}")


def build_scheduler(cfg: DentexConfig, optimizer: torch.optim.Optimizer) -> torch.optim.lr_scheduler.LambdaLR:
    """Create warmup + cosine decay scheduler by iteration."""
    total_iters = max(1, cfg.train_iterations)
    warmup_iters = max(1, cfg.warmup_steps)

    def lr_lambda(step: int) -> float:
        if step < warmup_iters:
            return float(step + 1) / float(warmup_iters)
        progress = (step - warmup_iters) / float(max(1, total_iters - warmup_iters))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return max(0.0, cosine)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def save_checkpoint(
    model: DentexPipeline,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LambdaLR,
    scaler: GradScaler,
    cfg: DentexConfig,
    iteration: int,
    val_ap: Optional[float] = None,
    is_best: bool = False,
) -> Path:
    """Save checkpoint locally and sync with W&B."""
    ckpt_name = f"dentex_iter_{iteration:06d}.pt" if not is_best else "dentex_best.pt"
    ckpt_path = Path(cfg.output_dir) / ckpt_name

    torch.save(
        {
            "iteration": iteration,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "cfg": asdict(cfg),
            "val_ap": val_ap,
        },
        ckpt_path,
    )

    if wandb.run is not None:
        wandb.save(str(ckpt_path))
    return ckpt_path


def train_supervised(
    model: DentexPipeline,
    loss_fn: DentexMultiTaskLoss,
    data_bundles: Dict[str, Any],
    cfg: DentexConfig,
    device: torch.device,
    eval_fn: Optional[Any] = None,
) -> Dict[str, Any]:
    """Run supervised training for configured iteration budget."""
    train_loader = data_bundles["train_loader"]
    val_loaders = data_bundles["val_loaders"]

    optimizer = build_optimizer(cfg, model)
    scheduler = build_scheduler(cfg, optimizer)
    amp_enabled = cfg.amp and device.type == "cuda"
    scaler = GradScaler(device=device.type, enabled=amp_enabled)

    best_ap = -float("inf")
    best_ckpt: Optional[Path] = None

    model.train()
    iteration = 0
    pbar = tqdm(total=cfg.train_iterations, desc="Supervised Training")

    while iteration < cfg.train_iterations:
        for batch in train_loader:
            images = batch["images"].to(device, non_blocking=True)
            targets = batch["targets"]
            for t in targets:
                t["boxes"] = t["boxes"].to(device)
                t["quadrants"] = t["quadrants"].to(device)
                if t["enumerations"] is not None:
                    t["enumerations"] = t["enumerations"].to(device)
                if t["masks"] is not None:
                    t["masks"] = t["masks"].to(device)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type=device.type, enabled=amp_enabled):
                outputs = model(images, targets)
                losses = loss_fn(outputs, targets)
                total_loss = losses["loss_total"]

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_max_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            iteration += 1
            pbar.update(1)

            log_payload = {
                "train/iter": iteration,
                "train/loss_total": float(losses["loss_total"].detach().cpu().item()),
                "train/loss_quadrant": float(losses["loss_quadrant"].detach().cpu().item()),
                "train/loss_seg": float(losses["loss_seg"].detach().cpu().item()),
                "train/loss_dino": float(losses["loss_dino"].detach().cpu().item()),
                "train/lr": float(optimizer.param_groups[0]["lr"]),
            }
            maybe_log_wandb(log_payload, step=iteration)

            pbar.set_postfix(
                total=f"{log_payload['train/loss_total']:.4f}",
                quad=f"{log_payload['train/loss_quadrant']:.4f}",
                seg=f"{log_payload['train/loss_seg']:.4f}",
                dino=f"{log_payload['train/loss_dino']:.4f}",
            )

            if iteration % cfg.ckpt_every_iters == 0:
                save_checkpoint(model, optimizer, scheduler, scaler, cfg, iteration, val_ap=None, is_best=False)

            if eval_fn is not None and iteration % cfg.val_every_iters == 0:
                model.eval()
                val_metrics = eval_fn(model, val_loaders, cfg, device)
                model.train()

                val_ap = float(val_metrics.get("enumeration_AP", 0.0))
                maybe_log_wandb({f"val/{k}": v for k, v in val_metrics.items()}, step=iteration)

                if val_ap > best_ap:
                    best_ap = val_ap
                    best_ckpt = save_checkpoint(
                        model,
                        optimizer,
                        scheduler,
                        scaler,
                        cfg,
                        iteration,
                        val_ap=best_ap,
                        is_best=True,
                    )

                del val_metrics
                torch.cuda.empty_cache()

            if iteration >= cfg.train_iterations:
                break

    pbar.close()

    return {
        "best_ap": best_ap,
        "best_ckpt": str(best_ckpt) if best_ckpt is not None else None,
        "final_iteration": iteration,
    }


if WANDB_READY and wandb.run is None:
    wandb.init(project=CFG.wandb_project, name=CFG.wandb_run_name, config=asdict(CFG))


## 14. Postprocessing: Box/Mask Fusion and FDI Assignment

This section converts segmentation outputs to boxes, fuses boxes with DINO detections, and enforces unique FDI IDs with Hungarian matching.


In [ ]:
FDI_IDS: List[int] = [
    11, 12, 13, 14, 15, 16, 17, 18,
    21, 22, 23, 24, 25, 26, 27, 28,
    31, 32, 33, 34, 35, 36, 37, 38,
    41, 42, 43, 44, 45, 46, 47, 48,
]
FDI_TO_CLASS_IDX: Dict[int, int] = {fdi: idx + 1 for idx, fdi in enumerate(FDI_IDS)}


def map_local_to_global_xyxy(local_box: Sequence[float], quad_box_global_xyxy: Sequence[float]) -> List[float]:
    """Map local normalized box to global normalized coordinates."""
    lx1, ly1, lx2, ly2 = [float(v) for v in local_box]
    qx1, qy1, qx2, qy2 = [float(v) for v in quad_box_global_xyxy]
    qw = max(qx2 - qx1, CFG.eps)
    qh = max(qy2 - qy1, CFG.eps)

    gx1 = qx1 + lx1 * qw
    gy1 = qy1 + ly1 * qh
    gx2 = qx1 + lx2 * qw
    gy2 = qy1 + ly2 * qh
    return [gx1, gy1, gx2, gy2]


def normalize_xyxy_list(boxes_xyxy_abs: List[List[float]], img_w: int, img_h: int) -> List[List[float]]:
    """Normalize absolute xyxy boxes to [0,1]."""
    out = []
    for x1, y1, x2, y2 in boxes_xyxy_abs:
        out.append([
            float(x1) / float(img_w),
            float(y1) / float(img_h),
            float(x2) / float(img_w),
            float(y2) / float(img_h),
        ])
    return out


def denormalize_xyxy_list(boxes_xyxy_norm: np.ndarray, img_w: int, img_h: int) -> List[List[float]]:
    """Denormalize xyxy boxes in [0,1] to absolute pixel coordinates."""
    out = []
    for box in boxes_xyxy_norm:
        x1, y1, x2, y2 = box.tolist()
        out.append([
            x1 * img_w,
            y1 * img_h,
            x2 * img_w,
            y2 * img_h,
        ])
    return out


def masks_to_boxes(
    seg_prob_map: np.ndarray,
    quadrant_idx: int,
    quad_box_xyxy_norm: Sequence[float],
    cfg: DentexConfig,
) -> Tuple[List[List[float]], List[float], List[int]]:
    """Convert one quadrant segmentation map into global normalized boxes."""
    labeled = label(seg_prob_map > 0.5)
    boxes: List[List[float]] = []
    scores: List[float] = []
    labels: List[int] = []

    for region in regionprops(labeled):
        if region.area < cfg.min_component_area:
            continue

        min_r, min_c, max_r, max_c = region.bbox
        hgt, wdt = seg_prob_map.shape

        local = [
            min_c / max(1, wdt),
            min_r / max(1, hgt),
            max_c / max(1, wdt),
            max_r / max(1, hgt),
        ]
        global_box = map_local_to_global_xyxy(local, quad_box_xyxy_norm)
        boxes.append(global_box)

        region_mask = (labeled[min_r:max_r, min_c:max_c] == region.label).astype(np.float32)
        region_score = float((seg_prob_map[min_r:max_r, min_c:max_c] * region_mask).sum() / max(cfg.eps, region_mask.sum()))
        scores.append(region_score)
        labels.append(quadrant_idx + 1)

    return boxes, scores, labels


def fuse_boxes_wbf(
    seg_boxes: List[List[float]],
    dino_boxes: List[List[float]],
    seg_scores: List[float],
    dino_scores: List[float],
    seg_labels: List[int],
    dino_labels: List[int],
    cfg: DentexConfig,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Fuse segmentation and DINO boxes using weighted box fusion."""
    if len(seg_boxes) == 0 and len(dino_boxes) == 0:
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.int64)

    boxes_list = [seg_boxes, dino_boxes]
    scores_list = [seg_scores, dino_scores]
    labels_list = [seg_labels, dino_labels]

    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        boxes_list=boxes_list,
        scores_list=scores_list,
        labels_list=labels_list,
        iou_thr=cfg.wbf_iou_thresh,
        skip_box_thr=cfg.wbf_score_thresh,
    )
    return fused_boxes, fused_scores, fused_labels


def assign_fdi_ids(
    fused_boxes_global_norm: List[List[float]],
    fused_quadrants: List[int],
    class_logits_by_box: np.ndarray,
    cfg: DentexConfig,
) -> List[Dict[str, Any]]:
    """Assign unique per-quadrant FDI IDs via Hungarian matching."""
    final_annotations: List[Dict[str, Any]] = []
    if len(fused_boxes_global_norm) == 0:
        return final_annotations

    quadrants_arr = np.array(fused_quadrants, dtype=np.int64)

    for quadrant in range(1, cfg.num_quadrants + 1):
        q_indices = np.where(quadrants_arr == quadrant)[0]
        if len(q_indices) == 0:
            continue

        q_logits = class_logits_by_box[q_indices]
        q_probs = torch.softmax(torch.from_numpy(q_logits), dim=-1).numpy()[:, : cfg.num_positions_per_quadrant]
        cost = 1.0 - q_probs

        row_idx, col_idx = linear_sum_assignment(cost)

        for rr, cc in zip(row_idx.tolist(), col_idx.tolist()):
            global_idx = int(q_indices[rr])
            fdi_id = quadrant * 10 + (cc + 1)
            final_annotations.append(
                {
                    "bbox_norm_xyxy": fused_boxes_global_norm[global_idx],
                    "fdi_id": fdi_id,
                    "quadrant": quadrant,
                    "score": float(q_probs[rr, cc]),
                }
            )

    return final_annotations


def postprocess_single_image(
    outputs: Dict[str, Any],
    image_index: int,
    cfg: DentexConfig,
) -> Dict[str, Any]:
    """Postprocess one image from model outputs into final annotations."""
    quad_boxes = outputs["quad_boxes_xyxy"][image_index].detach().cpu().numpy()

    seg_boxes_all: List[List[float]] = []
    seg_scores_all: List[float] = []
    seg_labels_all: List[int] = []

    dino_boxes_all: List[List[float]] = []
    dino_scores_all: List[float] = []
    dino_labels_all: List[int] = []
    dino_logits_all: List[np.ndarray] = []

    for q_idx in range(cfg.num_quadrants):
        q_box = quad_boxes[q_idx].tolist()

        seg_logits = outputs["seg_logits_by_quad"][q_idx][image_index, 0].detach().cpu().numpy()
        seg_prob = 1.0 / (1.0 + np.exp(-seg_logits))
        q_seg_boxes, q_seg_scores, q_seg_labels = masks_to_boxes(seg_prob, q_idx, q_box, cfg)
        seg_boxes_all.extend(q_seg_boxes)
        seg_scores_all.extend(q_seg_scores)
        seg_labels_all.extend(q_seg_labels)

        dino_boxes_local = outputs["dino_boxes_by_quad"][q_idx][image_index].detach().cpu().numpy()
        dino_logits = outputs["dino_logits_by_quad"][q_idx][image_index].detach().cpu().numpy()
        dino_probs = torch.softmax(torch.from_numpy(dino_logits), dim=-1).numpy()

        for i in range(dino_boxes_local.shape[0]):
            cls_id = int(np.argmax(dino_probs[i]))
            cls_score = float(np.max(dino_probs[i]))
            if cls_id == cfg.num_positions_per_quadrant:
                continue

            local_xyxy = cxcywh_to_xyxy(torch.tensor(dino_boxes_local[i : i + 1], dtype=torch.float32)).numpy()[0].tolist()
            global_xyxy = map_local_to_global_xyxy(local_xyxy, q_box)

            dino_boxes_all.append(global_xyxy)
            dino_scores_all.append(cls_score)
            dino_labels_all.append(q_idx + 1)
            dino_logits_all.append(dino_logits[i])

    fused_boxes, fused_scores, fused_labels = fuse_boxes_wbf(
        seg_boxes_all,
        dino_boxes_all,
        seg_scores_all,
        dino_scores_all,
        seg_labels_all,
        dino_labels_all,
        cfg,
    )

    if len(dino_logits_all) == 0:
        class_logits = np.zeros((len(fused_boxes), cfg.dino_num_output_classes), dtype=np.float32)
    else:
        base_logits = np.array(dino_logits_all, dtype=np.float32)
        if len(base_logits) < len(fused_boxes):
            pad = np.zeros((len(fused_boxes) - len(base_logits), cfg.dino_num_output_classes), dtype=np.float32)
            class_logits = np.concatenate([base_logits, pad], axis=0)
        else:
            class_logits = base_logits[: len(fused_boxes)]

    fused_boxes_list = [b.tolist() for b in fused_boxes]
    fused_quadrants = [int(lbl) for lbl in fused_labels.tolist()] if len(fused_labels) else []

    final_ann = assign_fdi_ids(
        fused_boxes_global_norm=fused_boxes_list,
        fused_quadrants=fused_quadrants,
        class_logits_by_box=class_logits,
        cfg=cfg,
    )

    return {
        "quad_boxes_norm_xyxy": quad_boxes.tolist(),
        "fused_boxes_norm_xyxy": fused_boxes_list,
        "fused_scores": fused_scores.tolist() if len(fused_scores) else [],
        "fused_quadrants": fused_quadrants,
        "final_annotations": final_ann,
    }


## 15. Evaluation Pipeline (COCO AP/AR + Reference Comparison)

This section computes AP and AR metrics and compares them against DENTEX benchmark references.


In [ ]:
def to_coco_bbox_from_norm_xyxy(box_norm_xyxy: Sequence[float], cfg: DentexConfig) -> List[float]:
    """Convert normalized xyxy box to absolute COCO [x,y,w,h]."""
    x1, y1, x2, y2 = box_norm_xyxy
    x_abs = float(x1) * cfg.img_w
    y_abs = float(y1) * cfg.img_h
    w_abs = max(cfg.eps, (float(x2) - float(x1)) * cfg.img_w)
    h_abs = max(cfg.eps, (float(y2) - float(y1)) * cfg.img_h)
    return [x_abs, y_abs, w_abs, h_abs]


def run_coco_eval(
    coco_gt: COCO,
    predictions: List[Dict[str, Any]],
    cfg: DentexConfig,
    use_cats: bool,
) -> Dict[str, float]:
    """Run COCOeval and return key AP/AR metrics."""
    if len(predictions) == 0:
        return {
            "AP": 0.0,
            "AP50": 0.0,
            "AP75": 0.0,
            "AR": 0.0,
        }

    coco_dt = coco_gt.loadRes(predictions)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.params.iouThrs = np.array(cfg.iou_thresholds)
    coco_eval.params.useCats = 1 if use_cats else 0
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    stats = coco_eval.stats
    return {
        "AP": float(stats[0]),
        "AP50": float(stats[1]),
        "AP75": float(stats[2]),
        "AR": float(stats[8]),
    }


def aggregate_metrics(metric_list: List[Dict[str, float]]) -> Dict[str, float]:
    """Average a list of metric dicts."""
    if len(metric_list) == 0:
        return {"AP": 0.0, "AP50": 0.0, "AP75": 0.0, "AR": 0.0}

    keys = metric_list[0].keys()
    return {k: float(np.mean([m[k] for m in metric_list])) for k in keys}


def get_reference_benchmark_metrics() -> Dict[str, float]:
    """Return DENTEX rank-1 reference values for comparison."""
    return {
        "quadrant_AR_ref": 0.754,
        "quadrant_AP_ref": 0.4745,
        "quadrant_AP50_ref": 0.6787,
        "quadrant_AP75_ref": 0.5846,
        "enumeration_AP_ref": 0.3535,
        "enumeration_AP50_ref": 0.5106,
        "enumeration_AP75_ref": 0.4207,
    }


def evaluate_model(
    model: DentexPipeline,
    val_loaders: Dict[str, DataLoader],
    cfg: DentexConfig,
    device: torch.device,
) -> Dict[str, float]:
    """Evaluate model on validation loaders for quadrant and enumeration metrics."""
    model.eval()
    quadrant_metrics_per_subset: List[Dict[str, float]] = []
    enum_metrics_per_subset: List[Dict[str, float]] = []

    with torch.no_grad():
        for subset_name, loader in val_loaders.items():
            quad_preds: List[Dict[str, Any]] = []
            enum_preds: List[Dict[str, Any]] = []

            for batch in tqdm(loader, desc=f"Eval {subset_name}", leave=False):
                images = batch["images"].to(device, non_blocking=True)
                outputs = model(images)
                quad_scores = outputs["quad_scores"].detach().cpu().numpy()

                for b_idx, target in enumerate(batch["targets"]):
                    image_id = int(target["image_id"].item())
                    sample_pp = postprocess_single_image(outputs, b_idx, cfg)

                    for q_idx, q_box in enumerate(sample_pp["quad_boxes_norm_xyxy"]):
                        score = float(quad_scores[b_idx, q_idx])
                        if score < cfg.wbf_score_thresh:
                            continue
                        quad_preds.append(
                            {
                                "image_id": image_id,
                                "category_id": q_idx + 1,
                                "bbox": to_coco_bbox_from_norm_xyxy(q_box, cfg),
                                "score": score,
                            }
                        )

                    for ann in sample_pp["final_annotations"]:
                        enum_preds.append(
                            {
                                "image_id": image_id,
                                "category_id": FDI_TO_CLASS_IDX.get(int(ann["fdi_id"]), 1),
                                "bbox": to_coco_bbox_from_norm_xyxy(ann["bbox_norm_xyxy"], cfg),
                                "score": float(ann["score"]),
                            }
                        )

            coco_gt = loader.dataset.coco
            subset_quad_metrics = run_coco_eval(coco_gt, quad_preds, cfg, use_cats=False)
            quadrant_metrics_per_subset.append(subset_quad_metrics)

            if loader.dataset.label_level in {"quadrant_enumeration", "diagnosis"}:
                subset_enum_metrics = run_coco_eval(coco_gt, enum_preds, cfg, use_cats=False)
                enum_metrics_per_subset.append(subset_enum_metrics)

    quad = aggregate_metrics(quadrant_metrics_per_subset)
    enum = aggregate_metrics(enum_metrics_per_subset)
    ref = get_reference_benchmark_metrics()

    metrics = {
        "quadrant_AP": quad["AP"],
        "quadrant_AP50": quad["AP50"],
        "quadrant_AP75": quad["AP75"],
        "quadrant_AR": quad["AR"],
        "enumeration_AP": enum["AP"],
        "enumeration_AP50": enum["AP50"],
        "enumeration_AP75": enum["AP75"],
        "enumeration_AR": enum["AR"],
        "quadrant_AP_gap_to_ref": quad["AP"] - ref["quadrant_AP_ref"],
        "quadrant_AP50_gap_to_ref": quad["AP50"] - ref["quadrant_AP50_ref"],
        "quadrant_AP75_gap_to_ref": quad["AP75"] - ref["quadrant_AP75_ref"],
        "quadrant_AR_gap_to_ref": quad["AR"] - ref["quadrant_AR_ref"],
        "enumeration_AP_gap_to_ref": enum["AP"] - ref["enumeration_AP_ref"],
        "enumeration_AP50_gap_to_ref": enum["AP50"] - ref["enumeration_AP50_ref"],
        "enumeration_AP75_gap_to_ref": enum["AP75"] - ref["enumeration_AP75_ref"],
    }

    if wandb.run is not None:
        table = wandb.Table(columns=["metric", "value"])
        for key, value in metrics.items():
            table.add_data(key, float(value))
        wandb.log({"evaluation/metrics_table": table, **{f"evaluation/{k}": v for k, v in metrics.items()}})

    return metrics


## 16. Inference and Visualization

This section runs single-image inference and visualizes quadrant-colored boxes with FDI IDs and confidence.


In [ ]:
QUADRANT_COLORS = {
    1: "blue",
    2: "green",
    3: "orange",
    4: "red",
}


def load_checkpoint_into_model(model: DentexPipeline, checkpoint_path: str, device: torch.device) -> None:
    """Load model weights from local checkpoint path."""
    state = torch.load(checkpoint_path, map_location=device)
    if "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"], strict=False)
    else:
        model.load_state_dict(state, strict=False)
    print(f"Loaded checkpoint from: {checkpoint_path}")


def preprocess_image_for_inference(image_path: str, cfg: DentexConfig) -> Tuple[torch.Tensor, np.ndarray]:
    """Load and preprocess one image for inference."""
    image_rgb = read_grayscale_as_rgb(Path(image_path))

    infer_tf = A.Compose(
        [
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
            A.Resize(cfg.img_h, cfg.img_w),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]
    )

    transformed = infer_tf(image=image_rgb)
    image_t = transformed["image"].unsqueeze(0)
    return image_t, image_rgb


def draw_predictions_on_image(
    image_rgb: np.ndarray,
    predictions: List[Dict[str, Any]],
    title: str,
) -> plt.Figure:
    """Draw bbox + FDI predictions on image."""
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(image_rgb, cmap="gray")

    hgt, wdt = image_rgb.shape[:2]
    for ann in predictions:
        x1, y1, x2, y2 = ann["bbox_norm_xyxy"]
        x1 *= wdt
        y1 *= hgt
        x2 *= wdt
        y2 *= hgt

        quadrant = int(ann["quadrant"])
        color = QUADRANT_COLORS.get(quadrant, "white")

        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=2)
        ax.add_patch(rect)
        ax.text(
            x1,
            max(0.0, y1 - 6.0),
            f"FDI {ann['fdi_id']} ({ann['score']:.2f})",
            color=color,
            fontsize=9,
            bbox={"facecolor": "black", "alpha": 0.5, "pad": 2},
        )

    ax.set_title(title)
    ax.axis("off")
    return fig


def run_single_image_inference(
    model: DentexPipeline,
    image_path: str,
    cfg: DentexConfig,
    device: torch.device,
) -> Dict[str, Any]:
    """Run inference and return postprocessed outputs for one image."""
    model.eval()
    image_t, image_rgb = preprocess_image_for_inference(image_path, cfg)
    image_t = image_t.to(device)

    with torch.no_grad():
        outputs = model(image_t)

    pp = postprocess_single_image(outputs, image_index=0, cfg=cfg)

    fig = draw_predictions_on_image(
        image_rgb=image_rgb,
        predictions=pp["final_annotations"],
        title="DENTEX Inference: FDI Assignment",
    )

    if wandb.run is not None:
        wandb.log({"inference/visualization": wandb.Image(fig)})

    return {
        "postprocessed": pp,
        "figure": fig,
    }


print("Inference helpers are ready.")


## 17. Notebook Reliability Checks and Profile Escalation

Run quick-profile smoke checks, then use the final cell to switch to full profile for benchmark-length runs.


In [ ]:
def run_quick_smoke_checks(
    cfg: DentexConfig,
    data_bundles: Dict[str, Any],
    model: DentexPipeline,
    device: torch.device,
) -> None:
    """Run fast checks for import/path/model/data integrity."""
    print("Running quick smoke checks...")

    assert Path(cfg.data_root).exists(), f"Data root missing: {cfg.data_root}"
    assert Path(cfg.output_dir).exists(), f"Output dir missing: {cfg.output_dir}"

    set_seed(cfg.seed)
    a = torch.rand(1).item()
    set_seed(cfg.seed)
    b = torch.rand(1).item()
    assert abs(a - b) < cfg.eps, "Determinism check failed for seeded random state."

    train_loader = data_bundles["train_loader"]
    batch = next(iter(train_loader))
    images = batch["images"].to(device)
    targets = batch["targets"]

    model.train()
    with torch.no_grad():
        outputs = model(images, targets)

    bsz = images.shape[0]
    assert outputs["quad_boxes_xyxy"].shape == (bsz, cfg.num_quadrants, 4)
    assert outputs["quad_logits"].shape == (bsz, cfg.num_quadrants)
    assert len(outputs["seg_logits_by_quad"]) == cfg.num_quadrants
    assert len(outputs["dino_boxes_by_quad"]) == cfg.num_quadrants
    assert len(outputs["dino_logits_by_quad"]) == cfg.num_quadrants

    pp = postprocess_single_image(outputs, image_index=0, cfg=cfg)
    assert isinstance(pp["final_annotations"], list)

    print("Quick smoke checks passed.")


# Execute reliability checks in quick profile.
run_quick_smoke_checks(CFG, DATA_BUNDLES, MODEL, DEVICE)


# Switch to full benchmark profile when needed.
CFG_FULL = apply_runtime_profile(DentexConfig(), "full")
print("Full profile is prepared. Set CFG = CFG_FULL and rebuild data/model to run benchmark-length training.")


# Optional end-to-end execution template (manual control):
# 1) simmim_ckpt = pretrain_simmim(MODEL, DATA_BUNDLES["unlabeled_loader"], CFG, DEVICE)
# 2) if simmim_ckpt is not None: load_checkpoint_into_model(MODEL, str(simmim_ckpt), DEVICE)
# 3) train_results = train_supervised(MODEL, LOSS_FN, DATA_BUNDLES, CFG, DEVICE, eval_fn=evaluate_model)
# 4) print(train_results)


## 18. Training Execution (Quick or Full)

Use this final section to run the full pipeline end-to-end.

- Set `RUN_PROFILE` to `"quick"` for a fast sanity run or `"full"` for benchmark-length training.
- Toggle SimMIM, supervised training, and evaluation independently.
- This cell rebuilds config, transforms, loaders, model, and loss so it stays state-safe.

In [ ]:
# Section 18: End-to-end execution controller

RUN_PROFILE = "quick"  # "quick" or "full"
RUN_SIMMIM = False
RUN_SUPERVISED = True
RUN_EVAL = False

assert RUN_PROFILE in {"quick", "full"}, "RUN_PROFILE must be 'quick' or 'full'."

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Rebuild state explicitly for the selected profile to avoid stale notebook objects.
CFG = apply_runtime_profile(DentexConfig(), RUN_PROFILE)
set_seed(CFG.seed)
ensure_dir(CFG.output_dir)

TRAIN_TRANSFORMS = build_train_transforms(CFG)
VAL_TRANSFORMS = build_val_transforms(CFG)
DATA_BUNDLES = build_datasets_and_loaders(CFG)
MODEL = DentexPipeline(CFG).to(DEVICE)
LOSS_FN = DentexMultiTaskLoss(CFG)

print(f"Running profile: {RUN_PROFILE}")
print(f"Device: {DEVICE}")
print(f"Unlabeled loader available: {DATA_BUNDLES['unlabeled_loader'] is not None}")

if WANDB_READY and wandb.run is None:
    wandb.init(project=CFG.wandb_project, name=f"{CFG.wandb_run_name}-{RUN_PROFILE}", config=asdict(CFG))
elif wandb.run is not None:
    wandb.config.update(asdict(CFG), allow_val_change=True)

simmim_ckpt = None
if RUN_SIMMIM:
    simmim_ckpt = pretrain_simmim(MODEL, DATA_BUNDLES["unlabeled_loader"], CFG, DEVICE)
    if simmim_ckpt is not None:
        load_checkpoint_into_model(MODEL, str(simmim_ckpt), DEVICE)

train_results = None
if RUN_SUPERVISED:
    train_results = train_supervised(
        MODEL,
        LOSS_FN,
        DATA_BUNDLES,
        CFG,
        DEVICE,
        eval_fn=evaluate_model,
    )
    print("Train results:", train_results)

metrics = None
if RUN_EVAL:
    metrics = evaluate_model(MODEL, DATA_BUNDLES["val_loaders"], CFG, DEVICE)
    print("Evaluation metrics:", metrics)

print("Execution cell finished.")